# Polarimetric signature: Python–C parity on ALOS-1

This notebook compares ten pixels of the San Francisco ALOS-1 scattering matrix using `polarimetric_signature` and C-PolSARpro `Polar_Signature.exe`. Python reads the NetCDF product; C reads the matching PolSARpro S2 files. The C routine accepts each pixel column and row over standard input after `plot`.

The C writer saves each **linear** signature divided by its own maximum. Python returns raw power, so the comparison normalizes each Python surface the same way. The C text headers also record the unnormalized maxima.

In [ ]:
from pathlib import Path
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from polsarpro.io import open_netcdf_beam
from polsarpro.polarisation import polarimetric_signature

c_executable = Path("/home/c_psp/Soft/bin/data_process_sngl/Polar_Signature.exe")
input_file = Path("/data/psp/test_files/SAN_FRANCISCO_ALOS1_slc.nc")
input_dir = Path("/data/psp/SAN_FRANCISCO_ALOS1")
output_dir = Path("/data/psp/res/polarimetric_signature_alos1_c")
output_dir.mkdir(parents=True, exist_ok=True)

points = {
    "P01": (1000, 200),
    "P02": (2500, 900),
    "P03": (4000, 400),
    "P04": (6000, 1000),
    "P05": (8000, 600),
    "P06": (9216, 624),
    "P07": (11000, 300),
    "P08": (13000, 850),
    "P09": (15000, 500),
    "P10": (17500, 1050),
}
n_phi, n_tau = 180, 90  # C's endpoint-inclusive grid

## Load the common input

The public Python defaults use exact 1° spacing (181 × 91 values). This parity run instead requests the C grid of 180 × 90 values. `row` and `col` are zero-based pixel positions, not geographic coordinates.

In [ ]:
S = open_netcdf_beam(input_file)
assert S.attrs["poltype"] == "S"
for row, col in points.values():
    assert 0 <= row < S.sizes["y"] and 0 <= col < S.sizes["x"]
print(S)

## Run Python and C-PolSARpro

The C executable is interactive. For each point, send `plot`, then **column**, **row**, output format `lin`, and `exit` through standard input. Its successful run returns status 1, so the notebook checks the `OKplotOK` acknowledgement and output files instead of the exit status.

In [ ]:
def read_c_signature(paths):
    values = np.fromfile(paths["bin"], dtype=np.float32)
    count_tau = int(values[0])
    tau_c = values[1 : 1 + count_tau]
    records = values[1 + count_tau :].reshape(-1, 1 + count_tau)
    phi_c = records[:, 0]
    surface = records[:, 1:]
    header = np.loadtxt(paths["txt"])
    assert int(header[0]) == count_tau and int(header[3]) == len(phi_c)
    return phi_c, tau_c, surface, float(header[9])


results = {}
for point, (row, col) in points.items():
    signature_py = polarimetric_signature(
        S, row=row, col=col, n_phi=n_phi, n_tau=n_tau
    )
    files = {
        name: {
            suffix: output_dir / f"{point.lower()}_{name}.{suffix}"
            for suffix in ("txt", "bin")
        }
        for name in ("copol", "xpol")
    }
    command = [
        str(c_executable),
        "-id", str(input_dir),
        "-iodf", "S2",
        "-fct", str(files["copol"]["txt"]),
        "-fcb", str(files["copol"]["bin"]),
        "-fxt", str(files["xpol"]["txt"]),
        "-fxb", str(files["xpol"]["bin"]),
    ]
    completed = subprocess.run(
        command,
        input=f"plot\n{col}\n{row}\nlin\nexit\n",
        text=True,
        capture_output=True,
        timeout=120,
        check=False,
    )
    assert "OKplotOK" in completed.stdout, (
        completed.returncode,
        completed.stdout,
        completed.stderr,
    )

    signature_c = {}
    for name, paths in files.items():
        phi_c, tau_c, surface, raw_max = read_c_signature(paths)
        assert surface.shape == (n_phi, n_tau)
        np.testing.assert_allclose(phi_c, signature_py.phi.values, atol=2e-5)
        np.testing.assert_allclose(tau_c, signature_py.tau.values, atol=2e-5)
        signature_c[name] = {"surface": surface, "raw_max": raw_max}
    results[point] = {"python": signature_py, "c": signature_c}

print(f"Compared {len(results)} points")

## Numerical comparison

The binary layout is one float32 tau count, the tau coordinates, then one record per phi: its coordinate followed by all tau values. Text-header line 10 (zero-based index 9) contains the raw maximum used to normalize that surface. Small numerical differences are expected because C stores its angle grid and intermediate terms in float32. The header uses six decimal places, so its raw maximum comparison includes a `5e-7` absolute tolerance for decimal rounding.

In [ ]:
rows = []
for point, result in results.items():
    row, col = points[point]
    for name in ("copol", "xpol"):
        raw_py = result["python"][name].values
        normalized_py = raw_py / raw_py.max()
        normalized_c = result["c"][name]["surface"]
        error = normalized_py - normalized_c
        rows.append(
            {
                "point": point,
                "surface": name,
                "row": row,
                "col": col,
                "max_abs": np.max(np.abs(error)),
                "mean_abs": np.mean(np.abs(error)),
                "rmse": np.sqrt(np.mean(error**2)),
                "python_raw_max": raw_py.max(),
                "c_raw_max_header": result["c"][name]["raw_max"],
            }
        )
        np.testing.assert_allclose(
            normalized_py, normalized_c, rtol=5e-5, atol=5e-6
        )
        np.testing.assert_allclose(
            raw_py.max(), result["c"][name]["raw_max"], rtol=1e-5, atol=5e-7
        )

metrics = pd.DataFrame(rows).set_index(["point", "surface"])
metrics

## Inspect one normalized comparison

The table above covers all ten pixels. This plot shows one representative point.

In [ ]:
point = "P06"
signature_py = results[point]["python"]
signature_c = results[point]["c"]

fig, axes = plt.subplots(2, 3, figsize=(13, 7), constrained_layout=True)
extent = [
    signature_py.tau.min(),
    signature_py.tau.max(),
    signature_py.phi.min(),
    signature_py.phi.max(),
]
for ax_row, name in zip(axes, ("copol", "xpol")):
    python_surface = signature_py[name].values / signature_py[name].values.max()
    c_surface = signature_c[name]["surface"]
    for ax, image, title in zip(
        ax_row,
        (python_surface, c_surface, python_surface - c_surface),
        ("Python", "C", "Python − C"),
    ):
        artist = ax.imshow(image, origin="lower", aspect="auto", extent=extent)
        ax.set(
            title=f"{point} {name}: {title}",
            xlabel="Tau (degrees)",
            ylabel="Phi (degrees)",
        )
        fig.colorbar(artist, ax=ax, shrink=0.75)
plt.show()